# Silver Label QA — Notebook 03

Inspects the output of `scripts/run_silver_labeling.py` before anything is
promoted from `data/silver_labels/` into the real `review_aspects` table.

Checks performed:
1. Coverage & volume — how many reviews got labels, how many failed, aspects-per-review distribution
2. Per-feature distribution — label counts and sentiment balance per `feature_key` (catches taxonomy categories that never fire, or ones that dominate)
3. Failure analysis — what kind of reviews the LLM failed to label
4. Random spot-check sample — for manual human read-through
5. Zero-aspect long-review audit — long reviews the LLM said had *no* relevant aspects, which either means recall is fine or the taxonomy is missing a category
6. Export a flagged sample for manual review before promotion

**Assumption:** this notebook assumes `run_silver_labeling.py` was run on the *full* `train.parquet` (no `--limit`). If you used `--limit`, update `ATTEMPTED_REVIEWS_PATH` logic in cell 2 accordingly, or the zero-aspect audit in section 5 will be wrong.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import matplotlib.pyplot as plt

from fixfirst.config.settings import settings

pd.set_option("display.max_colwidth", 120)

SILVER_DIR = settings.resolve_path(settings.data_silver_labels_dir)
PROCESSED_DIR = settings.resolve_path(settings.data_processed_dir)

labels_df = pd.read_parquet(SILVER_DIR / "silver_labels.parquet")
failures_df = pd.read_parquet(SILVER_DIR / "labeling_failures.parquet")

# Full set of reviews that were ATTEMPTED (input to the labeling run).
# This is what lets us distinguish "labeled as zero aspects" from "never attempted".
attempted_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")

print(f"labels_df: {len(labels_df)} aspect rows across {labels_df['review_id'].nunique()} reviews")
print(f"failures_df: {len(failures_df)} reviews failed labeling entirely")
print(f"attempted_df: {len(attempted_df)} reviews were input to the labeling run")

## 1. Coverage & volume

In [ ]:
attempted_ids = set(attempted_df["id"])
failed_ids = set(failures_df["review_id"])
aspect_ids = set(labels_df["review_id"])  # reviews with >=1 aspect label
zero_aspect_ids = attempted_ids - failed_ids - aspect_ids  # succeeded, but LLM found no aspects

total_attempted = len(attempted_ids)
failure_rate = len(failed_ids) / total_attempted if total_attempted else 0
zero_aspect_rate = len(zero_aspect_ids) / total_attempted if total_attempted else 0

print(f"Total attempted:        {total_attempted}")
print(f"Failed entirely:        {len(failed_ids)} ({failure_rate:.1%})")
print(f"Succeeded, zero aspects:{len(zero_aspect_ids)} ({zero_aspect_rate:.1%})")
print(f"Succeeded, >=1 aspect:  {len(aspect_ids)} ({len(aspect_ids)/total_attempted:.1%})")

if failure_rate > 0.05:
    print(f"\n[FLAG] Failure rate {failure_rate:.1%} exceeds 5% — inspect section 3 before trusting this labeling run.")

aspects_per_review = labels_df.groupby("review_id").size()
print("\nAspects-per-review distribution (reviews with >=1 aspect only):")
print(aspects_per_review.describe())

aspects_per_review.plot(kind="hist", bins=range(1, aspects_per_review.max() + 2), title="Aspects per review")
plt.xlabel("Number of aspects assigned")
plt.show()

## 2. Per-feature distribution

Two things to look for here:
- **Categories that never fire** (zero or near-zero count) — either genuinely rare in this data, or the LLM doesn't understand the category description well enough to recognize it. Worth rewording the `description` in `features_master` if so.
- **Sentiment imbalance** — a feature that's 95% negative with very few positive/neutral examples will be hard for the downstream fine-tuned classifier to learn well; may need targeted upsampling later.

In [ ]:
feature_sentiment = labels_df.groupby(["feature_key", "sentiment"]).size().unstack(fill_value=0)
feature_sentiment["total"] = feature_sentiment.sum(axis=1)
feature_sentiment = feature_sentiment.sort_values("total", ascending=False)
display(feature_sentiment)

never_fired = feature_sentiment[feature_sentiment["total"] == 0]
if len(never_fired):
    print(f"\n[FLAG] {len(never_fired)} taxonomy categories never appeared in any label: {list(never_fired.index)}")

feature_sentiment.drop(columns="total").plot(
    kind="bar", stacked=True, figsize=(10, 5), title="Sentiment breakdown per feature"
)
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3. Failure analysis

What kind of reviews is the LLM failing to produce parseable output for? Long reviews (context/truncation issues) and very short reviews (ambiguous, model unsure how to respond) are the usual suspects.

In [ ]:
if len(failures_df) == 0:
    print("No failures to analyze.")
else:
    failures_df = failures_df.copy()
    failures_df["word_count"] = failures_df["review_text"].str.split().str.len()

    print("Word count distribution of FAILED reviews:")
    print(failures_df["word_count"].describe())

    print("\nSample of failed reviews:")
    display(failures_df.sample(min(5, len(failures_df)), random_state=1)[["review_id", "word_count", "review_text"]])

## 4. Random spot-check sample

Manually read these against the review text — this is the actual QA step, not just a metric. Look for: wrong sentiment, missed aspects, hallucinated aspects that technically match a category but misread the review.

In [ ]:
N_SPOT_CHECK = 15

spot_check = (
    labels_df.groupby("review_id")
    .agg(review_text=("review_text", "first"), feature_key=("feature_key", list), sentiment=("sentiment", list))
    .sample(min(N_SPOT_CHECK, labels_df["review_id"].nunique()), random_state=7)
)

for review_id, row in spot_check.iterrows():
    print(f"[{review_id}]")
    print(f"  Review: {row['review_text']}")
    for fk, sent in zip(row["feature_key"], row["sentiment"]):
        print(f"    -> {fk}: {sent}")
    print()

## 5. Zero-aspect long-review audit

A short review like "Great app!" legitimately has no specific aspect to extract.
A 40-word review with **zero** aspects assigned is more suspicious — either the
review is genuinely generic praise/complaint with no specific feature mentioned,
or it discusses something the taxonomy doesn't have a category for yet.

In [ ]:
MIN_WORDS_TO_FLAG = 20

zero_aspect_df = attempted_df[attempted_df["id"].isin(zero_aspect_ids)].copy()
zero_aspect_df["word_count"] = zero_aspect_df["review_text"].str.split().str.len()

long_zero_aspect = zero_aspect_df[zero_aspect_df["word_count"] >= MIN_WORDS_TO_FLAG].sort_values(
    "word_count", ascending=False
)

print(f"{len(long_zero_aspect)} reviews are >= {MIN_WORDS_TO_FLAG} words but got zero aspects assigned.")
print("If a pattern emerges below (e.g. many reviews about the same undiscussed topic), add a features_master category for it.\n")

display(long_zero_aspect.head(10)[["id", "word_count", "review_text"]])

## 6. Export flagged sample for manual review

Combines the spot-check sample and the long zero-aspect reviews into one CSV a human annotator can go through and mark agree/disagree, before anything here is promoted into `review_aspects`.

In [ ]:
flagged_rows = []

for review_id, row in spot_check.iterrows():
    flagged_rows.append({
        "review_id": review_id,
        "review_text": row["review_text"],
        "llm_feature_keys": ";".join(row["feature_key"]),
        "llm_sentiments": ";".join(row["sentiment"]),
        "flag_reason": "random_spot_check",
        "human_agree": "",  # fill in: yes / no / partial
        "human_notes": "",
    })

for _, row in long_zero_aspect.iterrows():
    flagged_rows.append({
        "review_id": row["id"],
        "review_text": row["review_text"],
        "llm_feature_keys": "",
        "llm_sentiments": "",
        "flag_reason": "long_review_zero_aspects",
        "human_agree": "",
        "human_notes": "",
    })

flagged_df = pd.DataFrame(flagged_rows)
out_path = SILVER_DIR / "flagged_for_manual_review.csv"
flagged_df.to_csv(out_path, index=False)

print(f"Wrote {len(flagged_df)} rows to {out_path}")
print("Next step: open this CSV, fill in human_agree/human_notes, then use it to refine features_master")
print("and/or the prompt in src/fixfirst/labeling/prompt.py before training the fine-tuned model on this data.")